# Production Patterns

The capstone: error handling, retries, timeouts, cost estimation, and result validation that turn a working hybrid job into a production one.

**Objectives:**
- Estimate a job's total cost (instance + quantum) before you submit it
- Wrap transient device errors in a retry decorator with exponential backoff
- Validate a returned energy/result against physical bounds before trusting it
- Configure AwsQuantumJob.create(...) with a stopping_condition (maxRuntimeInSeconds) for cost control
- Compose all of the above into a single submit-validate-retrieve flow

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)
#
# This notebook teaches the Amazon Braket Hybrid Jobs SERVICE (AwsQuantumJob).
# It is NOT browser-runnable: it needs the real amazon-braket-sdk.
# Every executed cell below runs on the LocalSimulator with NO AWS credentials.
# The only AWS-touching code (AwsQuantumJob.create, log_metric, save_job_result)
# lives inside the entry-point STRING (never executed here) or behind RUN_ON_AWS=False.

import functools
import time
import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator

# These import cleanly without credentials; we do NOT call AWS at top level.
from braket.aws import AwsQuantumJob
from braket.jobs.metrics import log_metric
from braket.jobs import save_job_checkpoint, load_job_checkpoint, save_job_result

np.random.seed(7)

# Local simulator: free, instant, no credentials.
device = LocalSimulator()

# Master gate. Everything that would incur AWS cost is behind this flag.
RUN_ON_AWS = False

print("Setup complete. RUN_ON_AWS =", RUN_ON_AWS)

## 1. Cost estimation before submission

A Hybrid Job's bill is two streams added together: the classical **instance** (billed per hour for as long as the container runs) and the **quantum** tasks it submits (the same per-task and per-shot rates as standalone work -- a job gives you priority, not a discount).

The cheapest mistake is the one you catch before create(...). The function below takes the device rates, the shot count, the iteration count, the instance rate, and the expected runtime, and returns the estimated total. We use the GUIDE rates:

- Instance `ml.m5.large` ~ **$0.115/hour** (within the $0.10-$3.85/hr band)
- Managed simulator (SV1) iterations: no per-shot/per-task QPU charge in this estimate
- IonQ: **$0.30 per task + $0.01 per shot**

In [ ]:
def estimate_job_cost(*, instance_rate_per_hr, runtime_min,
                      per_task_usd, per_shot_usd, shots, iterations):
    """Pre-submission cost estimate for a hybrid job.

    Total = instance (rate * hours) + quantum (per iteration: one task + shots).
    Returns a dict so the caller can see each stream, not just the sum.
    """
    instance_cost = instance_rate_per_hr * (runtime_min / 60.0)
    quantum_cost = iterations * (per_task_usd + per_shot_usd * shots)
    return {
        "instance_cost": round(instance_cost, 4),
        "quantum_cost": round(quantum_cost, 4),
        "total": round(instance_cost + quantum_cost, 4),
    }


# A managed-simulator job: instance charge only (no per-shot QPU rate).
sim_estimate = estimate_job_cost(
    instance_rate_per_hr=0.115, runtime_min=10,
    per_task_usd=0.0, per_shot_usd=0.0,
    shots=1000, iterations=50,
)

# An IonQ-QPU job: instance charge PLUS per-task and per-shot quantum charges.
ionq_estimate = estimate_job_cost(
    instance_rate_per_hr=0.115, runtime_min=30,
    per_task_usd=0.30, per_shot_usd=0.01,
    shots=1000, iterations=50,
)

print("Managed-sim job (50 iters x 1000 shots):")
print(f"  instance ${sim_estimate['instance_cost']:>8.4f}")
print(f"  quantum  ${sim_estimate['quantum_cost']:>8.4f}")
print(f"  TOTAL    ${sim_estimate['total']:>8.4f}")
print()
print("IonQ-QPU job (50 iters x 1000 shots):")
print(f"  instance ${ionq_estimate['instance_cost']:>8.4f}")
print(f"  quantum  ${ionq_estimate['quantum_cost']:>8.4f}")
print(f"  TOTAL    ${ionq_estimate['total']:>8.4f}")
print()
print("Note: the QPU job costs ~%.0fx more -- prototype on the simulator first."
      % (ionq_estimate['total'] / max(sim_estimate['total'], 1e-9)))

In [ ]:
# Correctness check: the estimator must return exact arithmetic for a known input.
# instance = 1.0/hr * 1.0 hr            = 1.00
# quantum  = 10 * (0.30 + 0.01 * 100)   = 10 * 1.30 = 13.00
# total    =                              14.00
check = estimate_job_cost(
    instance_rate_per_hr=1.0, runtime_min=60,
    per_task_usd=0.30, per_shot_usd=0.01,
    shots=100, iterations=10,
)
assert check["instance_cost"] == 1.0, check
assert check["quantum_cost"] == 13.0, check
assert check["total"] == 14.0, check
print("Cost estimator arithmetic verified:", check)

## 2. Retries with exponential backoff

A job that runs for hours can fail for hours' worth of reasons: a throttled API call, a transient device error, a brief network blip. Many of these are *transient* -- the right response is to wait and try again, backing off so you do not hammer a struggling service.

The decorator below retries on a chosen exception class, doubling the delay each attempt. We demonstrate it on a **flaky local function** seeded to fail twice and then succeed on the third call, where the successful path runs a real Bell circuit on the LocalSimulator. No AWS involved -- the same decorator wraps device.run(...) in production.

In [ ]:
class TransientDeviceError(Exception):
    """Stand-in for a retryable Braket/device error (throttling, brief outage)."""


def retry_with_backoff(max_attempts=5, base_delay=0.01, backoff=2.0,
                       exceptions=(TransientDeviceError,)):
    """Retry fn on transient errors with exponential backoff.

    Only the listed exception types are retried; everything else propagates
    immediately (a bug should fail fast, not be retried 5 times).
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as err:
                    if attempt == max_attempts:
                        print(f"  attempt {attempt} failed; no retries left")
                        raise
                    print(f"  attempt {attempt} failed ({err}); "
                          f"retrying in {delay:.3f}s")
                    time.sleep(delay)
                    delay *= backoff
        return wrapper
    return decorator


# A flaky function: fails on calls 1 and 2, succeeds on call 3.
_call_log = {"count": 0}


@retry_with_backoff(max_attempts=5, base_delay=0.01)
def submit_bell_state():
    _call_log["count"] += 1
    if _call_log["count"] < 3:
        raise TransientDeviceError(f"throttled (call {_call_log['count']})")
    # Success path: a real circuit on the local simulator.
    bell = Circuit().h(0).cnot(0, 1)
    return device.run(bell, shots=500).result().measurement_counts


counts = submit_bell_state()
print(f"Succeeded after {_call_log['count']} calls. Counts: {dict(counts)}")

# Verify the flaky function eventually succeeded on exactly the 3rd attempt.
assert _call_log["count"] == 3, _call_log
assert set(counts.keys()) <= {"00", "11"}, "Bell state should only show 00 and 11"
print("Retry behavior verified: 2 transient failures, then success.")

## 3. Result validation after completion

A job that returns a number is not a job that returned a *correct* number. Before you write a ground-state energy into a report -- or feed it into the next stage of a pipeline -- sanity-check it against physical bounds. A finite electronic energy for a small molecule is negative and bounded; an optimizer that diverged, a misconfigured Hamiltonian, or a NaN will fall outside that window, and you want to catch it here, not three steps downstream.

The validator accepts a good result and rejects a bad one. We assert both directions.

In [ ]:
def validate_energy(energy, *, lower=-2.0, upper=0.0):
    """Sanity-check a returned energy (Hartree) against physical bounds.

    Returns (ok, reason). For a small molecule like H2 the ground-state
    electronic energy is finite and within roughly [-2, 0] Hartree; values
    outside that (or non-finite) signal a failed/diverged optimization.
    """
    if not np.isfinite(energy):
        return False, "non-finite energy (NaN/inf)"
    if not (lower <= energy <= upper):
        return False, f"energy {energy} outside physical bounds [{lower}, {upper}]"
    return True, "ok"


# A plausible H2 ground-state energy (good) vs. a diverged result (bad).
good_energy = -1.137
bad_energy = 5.0

good_ok, good_reason = validate_energy(good_energy)
bad_ok, bad_reason = validate_energy(bad_energy)

print(f"validate_energy({good_energy}) -> {good_ok} ({good_reason})")
print(f"validate_energy({bad_energy})  -> {bad_ok} ({bad_reason})")
print(f"validate_energy(nan)    -> {validate_energy(float('nan'))}")

# The validator must accept the good result and reject the bad one.
assert good_ok is True, "validator wrongly rejected a physical energy"
assert bad_ok is False, "validator wrongly accepted an out-of-bounds energy"
print("Validation logic verified: accepts good, rejects bad.")

## 4. A local stand-in for the job's inner loop

The inner loop of every hybrid job is the same shape: one fixed parameterized circuit, a fresh angle each iteration chosen by a classical optimizer, an objective read off the measurements. Here is that loop running locally on the simulator -- a FreeParameter circuit driven with inputs={...}, optimized by a crude scan, with its convergence plotted. This is the work the managed job would do; the next section just packages it.

In [ ]:
# A parameterized 2-qubit circuit (the literal inner loop of the job).
theta = FreeParameter("theta")
ansatz = Circuit().ry(0, theta).cnot(0, 1).probability()

# Toy objective: distance from a target |11> probability. The optimizer wants
# to drive P(11) toward 0.5 (a maximally entangled split).
target_p11 = 0.5


def objective(angle):
    result = device.run(ansatz, shots=2000, inputs={"theta": float(angle)}).result()
    probs = result.values[0]  # [P(00), P(01), P(10), P(11)]
    p11 = probs[3]
    return (p11 - target_p11) ** 2


# Coarse scan (a few iterations, fast).
angles = np.linspace(0.0, np.pi, 12)
history = []
best_angle, best_val = None, np.inf
for a in angles:
    val = objective(a)
    history.append(val)
    if val < best_val:
        best_val, best_angle = val, a

print(f"Best theta = {best_angle:.3f} rad, objective = {best_val:.5f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(angles, history, "o-", color="#3b6ea5")
ax.axvline(best_angle, ls="--", color="#aa3333", label=f"best = {best_angle:.2f}")
ax.set_xlabel("theta (rad)")
ax.set_ylabel("objective (lower is better)")
ax.set_title("Local variational scan (the job's inner loop)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Packaging it as a Hybrid Job

The local loop above is exactly what runs inside the managed container. To ship it, you write it as an **entry-point script** (the algorithm Braket runs on the instance) and hand that to AwsQuantumJob.create(...). Inside the script -- and only there, where a job context exists -- you call log_metric(...) to stream the convergence curve to CloudWatch and save_job_result(...) to persist the answer to S3.

We define the script as a **string** so the notebook stays credential-free: nothing in this cell executes the AWS code.

In [ ]:
# The algorithm script Braket would run on the managed instance.
# Defined as a string here -- NOT executed in the notebook (no job context, no creds).
# It folds in everything above: parametric compilation, metrics, validation, result.
ENTRY_POINT_SCRIPT = '''
import os
import numpy as np
from braket.aws import AwsDevice
from braket.circuits import Circuit, FreeParameter
from braket.jobs import save_job_result
from braket.jobs.metrics import log_metric


def validate_energy(energy, lower=-2.0, upper=0.0):
    return np.isfinite(energy) and (lower <= energy <= upper)


def main():
    device = AwsDevice(os.environ["AMZN_BRAKET_DEVICE_ARN"])
    theta = FreeParameter("theta")
    ansatz = Circuit().ry(0, theta).cnot(0, 1)

    best = None
    for step, angle in enumerate(np.linspace(0.0, np.pi, 20)):
        # Parametric compilation: same structure, only theta changes.
        result = device.run(ansatz, shots=1000, inputs={"theta": float(angle)}).result()
        energy = -float(result.measurement_counts.get("11", 0)) / 1000.0
        log_metric(metric_name="energy", value=energy, iteration_number=step)
        if best is None or energy < best:
            best = energy

    if not validate_energy(best):
        raise ValueError(f"Result failed validation: {best}")

    save_job_result({"final_energy": best})


if __name__ == "__main__":
    main()
'''

print(f"Entry-point script prepared ({len(ENTRY_POINT_SCRIPT)} chars). Not executed here.")
print("It calls log_metric / save_job_result, which only work inside a running job.")

In [ ]:
# Cost-controlled submission, gated behind RUN_ON_AWS.
#
# COST NOTE: AwsQuantumJob.create(...) spins up a billed instance
# (ml.m5.large ~ $0.115/hr) and, on a QPU, incurs per-task + per-shot charges.
# The stopping_condition's maxRuntimeInSeconds caps the job's wall-clock time. Run `make cost` to check current spend. Default OFF.

if RUN_ON_AWS:
    import tempfile
    import os

    # Estimate before you submit (Section 1).
    est = estimate_job_cost(
        instance_rate_per_hr=0.115, runtime_min=20,
        per_task_usd=0.0, per_shot_usd=0.0, shots=1000, iterations=20,
    )
    print(f"Estimated total before submit: ${est['total']:.4f}")

    # Write the entry-point string to a real file for create(...).
    script_dir = tempfile.mkdtemp()
    script_path = os.path.join(script_dir, "algorithm.py")
    with open(script_path, "w") as fh:
        fh.write(ENTRY_POINT_SCRIPT)

    job = AwsQuantumJob.create(
        device="arn:aws:braket:::device/quantum-simulator/amazon/sv1",
        source_module=script_path,
        entry_point="algorithm:main",
        instance_config={"instanceType": "ml.m5.large", "instanceCount": 1},
        # Cost control: halt the job when it exceeds its runtime budget (seconds).
        stopping_condition={"maxRuntimeInSeconds": 1800},
        wait_until_complete=False,
    )
    print("Submitted job:", job.arn)

    # Retrieve + validate after completion (would block on job state).
    # result = job.result()
    # ok, reason = validate_energy(result["final_energy"])
    # print("final energy:", result["final_energy"], "valid:", ok, reason)
else:
    print("RUN_ON_AWS is False -- no job submitted, no AWS cost incurred.")
    print("Set RUN_ON_AWS = True (with credentials) to create the job above.")

## 6. The whole flow, end to end

The four patterns compose into one disciplined submit path: **estimate**, **submit with retries**, **validate**, then act. Below is that flow run entirely on the local simulator -- estimate the cost, submit through the retry decorator, read an energy off the result, and validate it -- with no AWS involved.

In [ ]:
def run_validated_local_job(shots=2000):
    # 1. Estimate (a local sim run is free, but the habit is the point).
    est = estimate_job_cost(
        instance_rate_per_hr=0.115, runtime_min=1,
        per_task_usd=0.0, per_shot_usd=0.0, shots=shots, iterations=1,
    )
    print(f"[estimate] total ${est['total']:.4f}")

    # 2. Submit through the retry decorator.
    @retry_with_backoff(max_attempts=4, base_delay=0.01)
    def submit():
        angle = 2.0 * np.arccos(np.sqrt(0.93))
        c = Circuit().ry(0, angle).cnot(0, 1).probability()
        return device.run(c, shots=shots).result()

    result = submit()

    # 3. Read an energy-like quantity off the result.
    p11 = float(result.values[0][3])
    energy = -1.137 * p11  # toy mapping into Hartree-scale bounds
    print(f"[result]   P(11) = {p11:.3f} -> energy = {energy:.4f} Hartree")

    # 4. Validate before trusting it.
    ok, reason = validate_energy(energy)
    print(f"[validate] {'ACCEPTED' if ok else 'REJECTED'} ({reason})")
    if not ok:
        raise ValueError(f"Result failed validation: {reason}")
    return energy


final = run_validated_local_job()
print(f"\nValidated final energy: {final:.4f} Hartree")

## Exercises

Scaffolds below -- fill in the TODOs. All can be developed and tested on the local simulator.

In [ ]:
# Exercise 1: Cost a managed simulator by the minute.
# SV1 is billed per minute, not per shot. Write a variant of estimate_job_cost
# that takes a sim $/minute rate and compare an SV1 job to an IonQ job at the
# same iteration count.
#
# def estimate_sim_minute_cost(rate_per_min, runtime_min, instance_rate_per_hr):
#     # TODO: instance + sim-minute charge; return the same dict shape.
#     pass

# Exercise 2: Make the retry decorator jittered.
# Add random jitter to the backoff delay (e.g. delay * (1 + random.uniform(0, 0.5)))
# so many concurrent clients do not retry in lockstep ('thundering herd').
#
# TODO: import random; modify retry_with_backoff to add jitter.

# Exercise 3: Tighten the validator with a reference.
# Given a known good energy (e.g. -1.137 for H2 at equilibrium), reject any
# result that differs by more than a tolerance, not just out-of-bounds values.
#
# def validate_against_reference(energy, reference, tol=0.05):
#     # TODO: return (ok, reason)
#     pass

print("Exercises ready. Implement the TODOs above.")

## Summary

- **Estimate before you submit.** A job's bill is instance ($/hr) plus quantum (per-task + per-shot). The QPU estimate here was ~$515 vs. ~$0.02 on the simulator -- prototype on the simulator first.
- **Retry transient errors, fail fast on real bugs.** Exponential backoff on a chosen exception class; everything else propagates immediately.
- **Validate every returned result** against physical bounds before it flows downstream. Accept the good, reject the diverged or non-finite.
- **Cap cost at the source.** A stopping_condition's maxRuntimeInSeconds bounds the job's wall-clock, so a runaway optimization shuts itself down. It lives behind RUN_ON_AWS=False so the notebook stays free.
- **Compose them.** estimate -> submit-with-retries -> validate -> act is one disciplined path, demonstrated end to end on the local simulator with no credentials.

This is the capstone of the curriculum: from a single qubit in `00-prereqs` to a cost-controlled hybrid-job workflow -- estimate, submit with retries, validate, and bound the runtime -- packaged to run on managed infrastructure.

**Next:** Return to [`../GUIDE.md`](../GUIDE.md) for the full Hybrid Jobs reference, or revisit the curriculum index to point these tools at a problem worth solving.